In [29]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [30]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [32]:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [33]:
#aqa_judge_prompt, 
aqa_judge_instructions


"You are an expert evaluator. You will be given:\n1. A question from a student\n2. The original answer from the FAQ (ground truth)\n3. An answer generated by an AI assistant\n\nYour task is to decide if the AI answer is semantically equivalent to\nthe original answer.\n\nRules:\n- The AI answer does NOT need to be word-for-word identical\n- It should convey the same key information\n- Extra detail is fine as long as the core answer is correct\n- Mark 'bad' only if the AI answer is wrong or misses the key point\n\nBe fair and focus on correctness, not style."

In [ ]:
from openai import OpenAI
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress

openai_client = OpenAI(
    base_url="https://dashscope-intl.aliyuncs.com/compatible-mode/v1",
    api_key="your-key"
)

In [35]:
import pandas as pd

df_answers = pd.read_csv("../data/rag-answers-new.csv")
answers = df_answers.to_dict(orient="records")

In [39]:
rec = answers[4]
rec

{'question': 'Are you still taking final assignments for the credential?',
 'answer_llm': 'Based on the provided context, you can submit your project (the Capstone project, which is required for the certificate/credential) as long as the submission form is still open. If you want to receive a certificate, you need to submit your project while submissions are still being accepted. The context does not specify whether the form is currently open or closed right now.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [40]:
prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)
print(prompt)

Question:
Are you still taking final assignments for the credential?

Original Answer (ground truth):
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

AI Answer:
Based on the provided context, you can submit your project (the Capstone project, which is required for the certificate/credential) as long as the submission form is still open. If you want to receive a certificate, you need to submit your project while submissions are still being accepted. The context does not specify whether the form is currently open or closed right now.


In [41]:
eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
    model="glm-5.1"
)

In [42]:
eval_result

AnswerEvaluation(reasoning="The original answer directly answers the question with a 'Yes', confirming that final assignments are still being accepted. The AI answer, however, states that the context does not specify whether the form is currently open or closed, which contradicts the 'Yes' in the ground truth. By failing to confirm that submissions are still being accepted, the AI answer misses the key point of the original answer.", score='bad')

In [43]:
calc_price(usage)

{'input_cost': 3.7300000000000005e-05,
 'output_cost': 0.00021640000000000003,
 'total_cost': 0.00025370000000000004}

In [44]:
def evaluate_aqa(question, answer_orig, answer_llm, model="glm-5.1"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [46]:
rec = answers[1]
rec

{'question': 'If I enroll now, will I still get the cert?',
 'answer_llm': 'Yes, you can still get a certificate, provided you submit your project while submissions are still being accepted and you pass the Capstone project.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [47]:
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

In [50]:
eval_result.score, eval_result.reasoning

('good',
 "The AI answer conveys the same key information as the original answer, specifically that enrollment allows for certificate eligibility as long as the project is submitted while submissions are still open. The AI answer adds the detail about needing to pass the Capstone project, which is an extra true detail that doesn't contradict the core answer and aligns with the rules allowing extra detail.")

In [58]:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"],
        model="qwen3.7-max"
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning
    }

    return result, usage

In [ ]:
test_result = judge_record(rec)

In [56]:
test_result[0]

{'question': 'If I enroll now, will I still get the cert?',
 'document': '74eb249bbf',
 'score': 'good',
 'reasoning': 'The AI answer conveys the same core information as the original answer: yes, you can get a certificate if you enroll now, but you must submit your project while submissions are still open. The AI answer adds the extra detail that you must also pass the Capstone project, which is a logical requirement for receiving a certificate and does not contradict the original answer. The rules state that extra detail is fine as long as the core answer is correct.'}

In [57]:
len(answers)

435

In [ ]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers, judge_record)

In [ ]:
results[10]

In [ ]:
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

In [ ]:
calc_total_price(usages)

In [ ]:
df_eval = pd.DataFrame(evaluations)

In [ ]:
df_eval.head()

In [ ]:
df_eval.score.value_counts()

In [ ]:
df_eval.score.value_counts(normalize=True)

In [ ]:
df_eval[df_eval["score"] == "bad"].head()

In [ ]:
df_eval.to_csv("../data/rag-evaluations-new.csv", index=False)
